In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import requests
import os
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 数据下载与预处理
url = 'https://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
data_file = 'timemachine.txt'
if not os.path.exists(data_file):
    response = requests.get(url)
    with open(data_file, 'w', encoding='utf-8') as f:
        f.write(response.text)

with open(data_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    raw_text = ' '.join([line.strip().lower() for line in lines])
    raw_text = raw_text[:30000]   # 仅取前30000字符

vocab = sorted(list(set(raw_text)))
vocab_size = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for i, ch in enumerate(vocab)}
data_indices = torch.tensor([char_to_idx[ch] for ch in raw_text], dtype=torch.long)

class SeqDataset(Dataset):
    def __init__(self, indices, num_steps):
        self.indices = indices
        self.num_steps = num_steps
    def __len__(self):
        return len(self.indices) - self.num_steps
    def __getitem__(self, idx):
        return self.indices[idx:idx+self.num_steps], self.indices[idx+1:idx+self.num_steps+1]

print("数据准备完成，词汇表大小:", vocab_size)

Using device: cuda
数据准备完成，词汇表大小: 43


In [2]:
class RNNScratch(nn.Module):
    def __init__(self, vocab_size, num_hiddens):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.W_xh = nn.Parameter(torch.randn(vocab_size, num_hiddens) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * 0.01)
        self.b_h = nn.Parameter(torch.zeros(num_hiddens))
        self.W_hq = nn.Parameter(torch.randn(num_hiddens, vocab_size) * 0.01)
        self.b_q = nn.Parameter(torch.zeros(vocab_size))
    def init_state(self, batch_size, device):
        return (torch.zeros((batch_size, self.num_hiddens), device=device),)
    def forward(self, X, state):
        X_onehot = F.one_hot(X, self.vocab_size).float()
        X_onehot = X_onehot.permute(1, 0, 2)
        H, = state
        outputs = []
        for t in range(X_onehot.shape[0]):
            H = torch.tanh(torch.mm(X_onehot[t], self.W_xh) + torch.mm(H, self.W_hh) + self.b_h)
            Y = torch.mm(H, self.W_hq) + self.b_q
            outputs.append(Y)
        out = torch.cat(outputs, dim=0)
        return out, (H,)

class GRUModel(nn.Module):
    def __init__(self, vocab_size, num_hiddens):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.gru = nn.GRU(vocab_size, num_hiddens, batch_first=True)
        self.fc = nn.Linear(num_hiddens, vocab_size)
    def forward(self, X, state):
        X_onehot = F.one_hot(X, vocab_size).float()
        out, state = self.gru(X_onehot, state)
        out = self.fc(out)
        return out.reshape(-1, vocab_size), state
    def init_state(self, batch_size, device):
        return torch.zeros(1, batch_size, self.num_hiddens, device=device)

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, num_hiddens):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.lstm = nn.LSTM(vocab_size, num_hiddens, batch_first=True)
        self.fc = nn.Linear(num_hiddens, vocab_size)
    def forward(self, X, state):
        X_onehot = F.one_hot(X, vocab_size).float()
        out, state = self.lstm(X_onehot, state)
        out = self.fc(out)
        return out.reshape(-1, vocab_size), state
    def init_state(self, batch_size, device):
        return (torch.zeros(1, batch_size, self.num_hiddens, device=device),
                torch.zeros(1, batch_size, self.num_hiddens, device=device))

print("模型定义完成")

模型定义完成


In [3]:
def train_model(model, train_loader, num_epochs, lr, device, model_name):
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    ppl_history = []
    for epoch in range(1, num_epochs+1):
        total_loss = 0.0
        total_tokens = 0
        state = None
        epoch_start = time.time()
        pbar = tqdm(train_loader, desc=f'{model_name} Epoch {epoch}/{num_epochs}', leave=False)
        for X, Y in pbar:
            X, Y = X.to(device), Y.to(device)
            batch_size = X.shape[0]
            if state is None:
                state = model.init_state(batch_size, device)
            else:
                if isinstance(state, tuple):
                    state = tuple(s.detach() for s in state)
                else:
                    state = state.detach()
            y_hat, state = model(X, state)
            loss = loss_fn(y_hat, Y.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            total_loss += loss.item() * Y.numel()
            total_tokens += Y.numel()
            current_ppl = math.exp(total_loss / total_tokens) if total_tokens > 0 else 0
            pbar.set_postfix({'loss': f'{loss.item():.3f}', 'ppl': f'{current_ppl:.2f}'})
        epoch_ppl = math.exp(total_loss / total_tokens)
        ppl_history.append(epoch_ppl)
        epoch_time = time.time() - epoch_start
        print(f"  {model_name} Epoch {epoch}: ppl={epoch_ppl:.2f}, time={epoch_time:.1f}s")
    return ppl_history

In [ ]:
base_config = {
    'batch_size': 16,
    'num_steps': 25,
    'num_epochs': 15,
}
param_grid = [
    {'num_hiddens': 128, 'lr': 0.5},
    {'num_hiddens': 128, 'lr': 1.0},
    {'num_hiddens': 256, 'lr': 0.5},
    {'num_hiddens': 256, 'lr': 1.0},
]

best_ppl = float('inf')
best_params = None
dataset_tune = SeqDataset(data_indices, base_config['num_steps'])
loader_tune = DataLoader(dataset_tune, batch_size=base_config['batch_size'], shuffle=True, drop_last=True)

for params in param_grid:
    num_hiddens = params['num_hiddens']
    lr = params['lr']
    print(f"\n🚀 测试: hidden={num_hiddens}, lr={lr}")
    model = RNNScratch(vocab_size, num_hiddens).to(device)
    ppl_history = train_model(model, loader_tune, base_config['num_epochs'], lr, device, f"RNN(h{num_hiddens}_lr{lr})")
    final_ppl = ppl_history[-1]
    print(f"   最终困惑度: {final_ppl:.2f}")
    if final_ppl < best_ppl:
        best_ppl = final_ppl
        best_params = params
    # 清理显存
    del model
    torch.cuda.empty_cache()

print(f"\n✅ 调参完成，最佳参数: hidden={best_params['num_hiddens']}, lr={best_params['lr']} (困惑度={best_ppl:.2f})")


🚀 测试: hidden=128, lr=0.5


  RNN(h128_lr0.5) Epoch 1: ppl=19.45, time=47.7s


  RNN(h128_lr0.5) Epoch 2: ppl=19.21, time=47.4s


  RNN(h128_lr0.5) Epoch 3: ppl=19.21, time=41.0s


  RNN(h128_lr0.5) Epoch 4: ppl=19.20, time=44.1s


RNN(h128_lr0.5) Epoch 5/15:  66%|██████▌   | 1240/1873 [00:25<00:15, 40.84it/s, loss=2.916, ppl=19.19]

In [ ]:
final_epochs = 40
dataset_final = SeqDataset(data_indices, base_config['num_steps'])
loader_final = DataLoader(dataset_final, batch_size=base_config['batch_size'], shuffle=True, drop_last=True)

# 训练 RNN
print("\n训练 RNN (best params)")
rnn_model = RNNScratch(vocab_size, best_params['num_hiddens']).to(device)
rnn_hist = train_model(rnn_model, loader_final, final_epochs, best_params['lr'], device, "RNN (best)")
# 清理
del rnn_model
torch.cuda.empty_cache()

# 训练 GRU
print("\n训练 GRU (same params)")
gru_model = GRUModel(vocab_size, best_params['num_hiddens']).to(device)
gru_hist = train_model(gru_model, loader_final, final_epochs, best_params['lr'], device, "GRU")
del gru_model
torch.cuda.empty_cache()

# 训练 LSTM
print("\n训练 LSTM (same params)")
lstm_model = LSTMModel(vocab_size, best_params['num_hiddens']).to(device)
lstm_hist = train_model(lstm_model, loader_final, final_epochs, best_params['lr'], device, "LSTM")
del lstm_model
torch.cuda.empty_cache()


训练 RNN (best params)


  RNN (best) Epoch 1: ppl=19.44, time=47.0s


  RNN (best) Epoch 2: ppl=19.22, time=47.3s


  RNN (best) Epoch 3: ppl=19.20, time=46.2s


  RNN (best) Epoch 4: ppl=19.20, time=45.7s


...

  LSTM Epoch 36: ppl=1.92, time=14.7s


  LSTM Epoch 37: ppl=1.92, time=14.3s


  LSTM Epoch 38: ppl=1.91, time=14.9s


  LSTM Epoch 39: ppl=1.91, time=14.3s


  LSTM Epoch 40: ppl=1.90, time=14.5s


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, final_epochs+1), rnn_hist, label=f'RNN (hidden={best_params["num_hiddens"]})', linewidth=2)
plt.plot(range(1, final_epochs+1), gru_hist, label='GRU', linewidth=2)
plt.plot(range(1, final_epochs+1), lstm_hist, label='LSTM', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title(f'RNN vs GRU vs LSTM (best RNN params: hidden={best_params["num_hiddens"]}, lr={best_params["lr"]})')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('rnn_gru_lstm_final.png', dpi=150)
plt.show()

print("\n========== 最终困惑度 ==========")
print(f"RNN  : 最低={min(rnn_hist):.2f}, 最终={rnn_hist[-1]:.2f}")
print(f"GRU  : 最低={min(gru_hist):.2f}, 最终={gru_hist[-1]:.2f}")
print(f"LSTM : 最低={min(lstm_hist):.2f}, 最终={lstm_hist[-1]:.2f}")